# Fraud Detection Using Spark ML
**Dataset:** E-Commerce Fraud Detection Dataset  
**Model:** Random Forest Classifier (Binary Classification)  
**Target:** `is_fraud` (0 = bình thường, 1 = gian lận)  
**Môn:** Big Data

## 1. Khởi tạo Spark Session

In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, hour, dayofweek, when, unix_timestamp
)
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, VectorAssembler, StandardScaler
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

spark = SparkSession.builder \
    .appName("FraudDetection_SparkML") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark Session khởi tạo thành công!")
print(f"Spark version: {spark.version}")

Spark Session khởi tạo thành công!
Spark version: 4.1.1


## 2. Đọc dữ liệu từ HDFS

In [20]:
HDFS_PATH = "hdfs://localhost:9090/ecommerce-fraud-detection/transactions.csv"

df = spark.read.csv(
    HDFS_PATH,
    header=True,
    inferSchema=True
)

## 3. Khám phá dữ liệu (EDA)

In [21]:
print("PHÂN PHỐI NHÃN (is_fraud)")
df.groupBy("is_fraud").count() \
    .withColumnRenamed("count", "so_luong") \
    .show()

PHÂN PHỐI NHÃN (is_fraud)
+--------+--------+
|is_fraud|so_luong|
+--------+--------+
|       1|    6612|
|       0|  293083|
+--------+--------+



In [22]:
print("THỐNG KÊ MÔ TẢ")
df.select(
    "amount", "account_age_days",
    "total_transactions_user", "avg_amount_user",
    "shipping_distance_km"
).describe().show()

THỐNG KÊ MÔ TẢ


+-------+------------------+-----------------+-----------------------+------------------+--------------------+
|summary|            amount| account_age_days|total_transactions_user|   avg_amount_user|shipping_distance_km|
+-------+------------------+-----------------+-----------------------+------------------+--------------------+
|  count|            299695|           299695|                 299695|            299695|              299695|
|   mean|177.16527856654275|973.3978711690219|      50.67332120989673|148.14297288910456|  357.04902767813854|
| stddev| 306.9265069627867|525.2414091711919|      5.976391419460316|200.36462383229753|   427.6720736404952|
|    min|               1.0|                1|                     40|              3.52|                 0.0|
|    max|          16994.74|             1890|                     60|           4565.29|             3748.56|
+-------+------------------+-----------------+-----------------------+------------------+--------------------+



In [23]:
print("PHÂN PHỐI THEO CHANNEL")
df.groupBy("channel").count().orderBy("count", ascending=False).show()

print("PHÂN PHỐI THEO MERCHANT CATEGORY")
df.groupBy("merchant_category").count().orderBy("count", ascending=False).show()

PHÂN PHỐI THEO CHANNEL
+-------+------+
|channel| count|
+-------+------+
|    web|152226|
|    app|147469|
+-------+------+

PHÂN PHỐI THEO MERCHANT CATEGORY
+-----------------+-----+
|merchant_category|count|
+-----------------+-----+
|      electronics|60220|
|           travel|59922|
|          grocery|59913|
|           gaming|59839|
|          fashion|59801|
+-----------------+-----+



## 4. Feature Engineering
Tạo thêm các đặc trưng mới từ dữ liệu hiện có để cải thiện độ chính xác của model.

In [24]:
# Trích xuất giờ giao dịch từ transaction_time
df = df.withColumn(
    "transaction_hour",
    hour(col("transaction_time").cast("timestamp"))
)

# Trích xuất ngày trong tuần
df = df.withColumn(
    "transaction_dayofweek",
    dayofweek(col("transaction_time").cast("timestamp"))
)

# country_match: country == bin_country hay không
df = df.withColumn(
    "country_match",
    when(col("country") == col("bin_country"), 1).otherwise(0)
)

# is_weekend: giao dịch vào cuối tuần
df = df.withColumn(
    "is_weekend",
    when(col("transaction_dayofweek").isin([1, 7]), 1).otherwise(0)
)

# amount_ratio: tỉ lệ số tiền so với trung bình của user
df = df.withColumn(
    "amount_ratio",
    col("amount") / (col("avg_amount_user") + 1)
)

print("Feature engineering hoàn tất!")
print(f"Tổng số features sau engineering: {len(df.columns)}")
df.select(
    "transaction_hour", "transaction_dayofweek",
    "country_match", "is_weekend", "amount_ratio"
).show(5)

Feature engineering hoàn tất!
Tổng số features sau engineering: 22
+----------------+---------------------+-------------+----------+------------------+
|transaction_hour|transaction_dayofweek|country_match|is_weekend|      amount_ratio|
+----------------+---------------------+-------------+----------+------------------+
|              11|                    7|            1|         1|0.5690592895991405|
|               3|                    4|            1|         0|0.7245014436312361|
|              13|                    6|            1|         0|0.6201571207950043|
|               0|                    3|            1|         0|0.7551870006043107|
|               8|                    4|            0|         0|0.8924326864970119|
+----------------+---------------------+-------------+----------+------------------+
only showing top 5 rows


## 5. Chuẩn bị Features cho Training

In [25]:
# Encode categorical features sang dạng số
categorical_cols = ["country", "bin_country", "channel", "merchant_category"]
indexers = [
    StringIndexer(
        inputCol=col_name,
        outputCol=col_name + "_idx",
        handleInvalid="keep"
    )
    for col_name in categorical_cols
]

# Danh sách features đưa vào model
feature_cols = [
    # Numerical features
    "account_age_days",
    "total_transactions_user",
    "avg_amount_user",
    "amount",
    "shipping_distance_km",
    "amount_ratio",
    # Binary features
    "promo_used",
    "avs_match",
    "cvv_result",
    "three_ds_flag",
    "country_match",
    "is_weekend",
    # Time features
    "transaction_hour",
    "transaction_dayofweek",
    # Encoded categorical features
    "country_idx",
    "bin_country_idx",
    "channel_idx",
    "merchant_category_idx"
]

# Vector Assembler: gộp tất cả features thành 1 vector
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw"
)

# Standard Scaler: chuẩn hóa features
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=True
)

print(f"Tổng số features đưa vào model: {len(feature_cols)}")
print("Danh sách features:", feature_cols)

Tổng số features đưa vào model: 18
Danh sách features: ['account_age_days', 'total_transactions_user', 'avg_amount_user', 'amount', 'shipping_distance_km', 'amount_ratio', 'promo_used', 'avs_match', 'cvv_result', 'three_ds_flag', 'country_match', 'is_weekend', 'transaction_hour', 'transaction_dayofweek', 'country_idx', 'bin_country_idx', 'channel_idx', 'merchant_category_idx']


## 6. Xây dựng Model — Random Forest Classifier

In [26]:
rf = RandomForestClassifier(
    labelCol="is_fraud",
    featuresCol="features",
    numTrees=100,
    maxDepth=10,
    seed=42
)

print("Random Forest Classifier đã được khởi tạo!")
print(f"  numTrees : {rf.getNumTrees()}")
print(f"  maxDepth : {rf.getMaxDepth()}")

Random Forest Classifier đã được khởi tạo!
  numTrees : 100
  maxDepth : 10


## 7. Xây dựng Pipeline

In [27]:
pipeline = Pipeline(stages=indexers + [assembler, scaler, rf])

print("Pipeline stages:")
for i, stage in enumerate(pipeline.getStages()):
    print(f"  Stage {i+1}: {type(stage).__name__}")

Pipeline stages:
  Stage 1: StringIndexer
  Stage 2: StringIndexer
  Stage 3: StringIndexer
  Stage 4: StringIndexer
  Stage 5: VectorAssembler
  Stage 6: StandardScaler
  Stage 7: RandomForestClassifier


## 8. Chia dữ liệu Train / Test

In [28]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print("CHIA DỮ LIỆU")
print(f"Training set : {train_df.count():,} records (80%)")
print(f"Test set     : {test_df.count():,} records (20%)")

print("\nPhân phối fraud trong Training set:")
train_df.groupBy("is_fraud").count().show()

print("Phân phối fraud trong Test set:")
test_df.groupBy("is_fraud").count().show()

CHIA DỮ LIỆU


Training set : 239,995 records (80%)


Test set     : 59,700 records (20%)

Phân phối fraud trong Training set:


+--------+------+
|is_fraud| count|
+--------+------+
|       1|  5309|
|       0|234686|
+--------+------+

Phân phối fraud trong Test set:


+--------+-----+
|is_fraud|count|
+--------+-----+
|       1| 1303|
|       0|58397|
+--------+-----+



## 9. Training Model

In [29]:
print("Đang train Random Forest")
model = pipeline.fit(train_df)
print("Training hoàn tất")

Đang train Random Forest


26/06/10 19:38:10 WARN DAGScheduler: Broadcasting large task binary with size 1103.1 KiB
26/06/10 19:38:12 WARN DAGScheduler: Broadcasting large task binary with size 1886.0 KiB
26/06/10 19:38:15 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
26/06/10 19:38:18 WARN DAGScheduler: Broadcasting large task binary with size 5.0 MiB


Training hoàn tất


## 10. Dự đoán và Đánh giá

In [30]:
predictions = model.transform(test_df)

print("Một số kết quả dự đoán:")
predictions.select(
    "transaction_id",
    "amount",
    "is_fraud",
    "prediction",
    "probability"
).show(10, truncate=False)

Một số kết quả dự đoán:


26/06/10 19:38:24 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB


+--------------+------+--------+----------+------------------------------------------+
|transaction_id|amount|is_fraud|prediction|probability                               |
+--------------+------+--------+----------+------------------------------------------+
|3             |92.36 |0       |0.0       |[0.9985328324618555,0.0014671675381443942]|
|7             |125.98|0       |0.0       |[0.9985569214062114,0.001443078593788617] |
|9             |261.58|0       |0.0       |[0.9843003998650596,0.015699600134940334] |
|14            |255.65|0       |0.0       |[0.9962717330244462,0.0037282669755538374]|
|20            |239.53|0       |0.0       |[0.998129282821169,0.001870717178830961]  |
|24            |128.92|0       |0.0       |[0.9981244077443151,0.001875592255684987] |
|30            |110.29|0       |0.0       |[0.9979893396374119,0.0020106603625880473]|
|36            |153.14|0       |0.0       |[0.9980008397608965,0.0019991602391034495]|
|46            |120.12|0       |0.0       |

In [31]:
# AUC-ROC
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="is_fraud",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc = evaluator_auc.evaluate(predictions)

# Accuracy
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="is_fraud", predictionCol="prediction", metricName="accuracy"
)
accuracy = evaluator_acc.evaluate(predictions)

# Precision
evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="is_fraud", predictionCol="prediction", metricName="weightedPrecision"
)
precision = evaluator_precision.evaluate(predictions)

# Recall
evaluator_recall = MulticlassClassificationEvaluator(
    labelCol="is_fraud", predictionCol="prediction", metricName="weightedRecall"
)
recall = evaluator_recall.evaluate(predictions)

# F1-Score
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="is_fraud", predictionCol="prediction", metricName="f1"
)
f1 = evaluator_f1.evaluate(predictions)

print("=" * 50)
print("       KẾT QUẢ ĐÁNH GIÁ MODEL")
print("=" * 50)
print(f"  AUC-ROC   : {auc:.4f}")
print(f"  Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print("=" * 50)

26/06/10 19:38:25 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
26/06/10 19:38:29 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
26/06/10 19:38:31 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
26/06/10 19:38:34 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
26/06/10 19:38:36 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB


       KẾT QUẢ ĐÁNH GIÁ MODEL
  AUC-ROC   : 0.9720
  Accuracy  : 0.9907 (99.07%)
  Precision : 0.9902
  Recall    : 0.9907
  F1-Score  : 0.9897


## 11. Confusion Matrix

In [32]:
print("CONFUSION MATRIX")
predictions.groupBy("is_fraud", "prediction").count() \
    .orderBy("is_fraud", "prediction") \
    .show()

tp = predictions.filter((col("is_fraud") == 1) & (col("prediction") == 1)).count()
tn = predictions.filter((col("is_fraud") == 0) & (col("prediction") == 0)).count()
fp = predictions.filter((col("is_fraud") == 0) & (col("prediction") == 1)).count()
fn = predictions.filter((col("is_fraud") == 1) & (col("prediction") == 0)).count()

print(f"True Positive  (TP): {tp:,}  → Gian lận dự đoán đúng là gian lận")
print(f"True Negative  (TN): {tn:,}  → Bình thường dự đoán đúng là bình thường")
print(f"False Positive (FP): {fp:,}   → Bình thường nhưng dự đoán là gian lận")
print(f"False Negative (FN): {fn:,}   → Gian lận nhưng dự đoán là bình thường")

CONFUSION MATRIX


26/06/10 19:38:38 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
26/06/10 19:38:41 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB


+--------+----------+-----+
|is_fraud|prediction|count|
+--------+----------+-----+
|       0|       0.0|58339|
|       0|       1.0|   58|
|       1|       0.0|  500|
|       1|       1.0|  803|
+--------+----------+-----+



26/06/10 19:38:42 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
26/06/10 19:38:44 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
26/06/10 19:38:47 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
26/06/10 19:38:49 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB


True Positive  (TP): 803  → Gian lận dự đoán đúng là gian lận
True Negative  (TN): 58,339  → Bình thường dự đoán đúng là bình thường
False Positive (FP): 58   → Bình thường nhưng dự đoán là gian lận
False Negative (FN): 500   → Gian lận nhưng dự đoán là bình thường


## 12. Feature Importance
Xem đặc trưng nào ảnh hưởng nhiều nhất đến kết quả dự đoán.

In [34]:
rf_model = model.stages[-1]
feature_importances = [float(x) for x in rf_model.featureImportances.toArray()]

importance_df = spark.createDataFrame(
    sorted(
        zip(feature_cols, feature_importances),
        key=lambda x: x[1],
        reverse=True
    ),
    schema=["feature", "importance"]
)

print("TOP FEATURES QUAN TRỌNG NHẤT")
importance_df.show(10, truncate=False)

TOP FEATURES QUAN TRỌNG NHẤT


+--------------------+--------------------+
|feature             |importance          |
+--------------------+--------------------+
|shipping_distance_km|0.24737548719294697 |
|account_age_days    |0.22872431133483495 |
|amount_ratio        |0.17688883018116572 |
|amount              |0.07753445591371524 |
|avs_match           |0.060351706987310516|
|avg_amount_user     |0.054621992759985305|
|cvv_result          |0.04538273675479608 |
|country_match       |0.026377924355806062|
|three_ds_flag       |0.023834715915882525|
|channel_idx         |0.017852671224118575|
+--------------------+--------------------+
only showing top 10 rows


## 13. Diễn giải Kết quả

### Bài toán
Phát hiện giao dịch gian lận trong thương mại điện tử bằng **Random Forest Classifier** trên **Spark ML**.

### Ý nghĩa các chỉ số
| Chỉ số | Ý nghĩa |
|---|---|
| **AUC-ROC** | Khả năng phân biệt fraud/non-fraud. >0.9: Xuất sắc, >0.8: Tốt |
| **Accuracy** | Tỉ lệ dự đoán đúng tổng thể |
| **Precision** | Trong số giao dịch bị gắn cờ fraud, bao nhiêu % thực sự là fraud |
| **Recall** | Trong số giao dịch fraud thực tế, model phát hiện được bao nhiêu % |
| **F1-Score** | Trung bình điều hòa của Precision và Recall |

### Lưu ý quan trọng
> Với bài toán fraud detection, **Recall quan trọng hơn Precision** vì bỏ sót giao dịch gian lận (FN) gây thiệt hại lớn hơn cảnh báo nhầm (FP).

### Ứng dụng thực tế
- Ngân hàng và ví điện tử có thể tích hợp model này để **tự động chặn/cảnh báo** giao dịch đáng ngờ theo thời gian thực.
- Hệ thống có thể mở rộng sang **multi-node Spark cluster** để xử lý hàng triệu giao dịch mỗi ngày, đây là điểm mạnh cốt lõi của Big Data.

## 14. Lưu Model (tuỳ chọn)

In [35]:
# Bỏ comment dòng dưới nếu muốn lưu model vào HDFS
# model.save("hdfs://localhost:9090/models/fraud_detection_rf")
# print("Model đã được lưu vào HDFS!")

spark.stop()
print("Spark Session đã dừng. Hoàn tất!")

Spark Session đã dừng. Hoàn tất!
